# 使用marker + langchain 完整rag流程

In [22]:
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from marker.config.parser import ConfigParser
import os


# ------------------ 配置 ------------------
input_pdf_path = r"knowledgeBase\pdfParsed_input\《营造法式》解读 第2版术语库_test.pdf"

custom_prompt = """
你是一个专业的 PDF 解析辅助助手。
在处理文档时，请特别注意：
1. 保持数学公式的 LaTeX 格式准确性。
2. 如果遇到表格，请确保其结构严谨，不要丢失行列信息。
3. 如果原文中有代码块，请识别出编程语言并进行标准 Markdown 格式化。
"""
block_custom_prompt = """
你是一个资深的古建筑文献编辑，请对提取的文本块进行以下处理：
1. 修正 OCR 识别的语义错误和基础错别字，但是请不要修改任何专业术语的字和词，我后续会人工校验。
2. 移除任何残留的乱码或不属于正文的字符如HTML标签<i></i> 、<br>。
3. 调整markdown格式错误，比如错误的层级，多余的加粗等。
输出格式要求：
不要输出任何开场白，直接输出修正后的文字内容，不要包含任何解释。
"""

config = {
    "output_format": "markdown",
    "paginate_output": True,
    
    "use_llm": True,
    "llm_service": "marker.services.openai.OpenAIService", 
    "openai_api_key": os.getenv("SEUAI_API_KEY"), 
    "openai_base_url": "http://10.128.202.100:3010/v1",
    "openai_model": "qwen3-vl-plus-2025-09-23",
    # "llm_prompt": custom_prompt,  # 传入自定义 Prompt,处理布局问题
    "block_correction_prompt": block_custom_prompt, # 对识别到的block文本根据自定义 Prompt 进行修正
}

config_parser = ConfigParser(config)
converter = PdfConverter(
    config = config_parser.generate_config_dict(),
    artifact_dict = create_model_dict(),
    llm_service=config_parser.get_llm_service()
)

rendered = converter(input_pdf_path)
text, _, images = text_from_rendered(rendered)

print("转换完成，输出文本前 400 字：")
print(text[:400])


LLMTableProcessor running: 100%|██████████| 1/1 [00:52<00:00, 52.17s/it]
LLM processors running: 0it [00:00, ?it/s]
Running LLMSectionHeaderProcessor: 100%|██████████| 1/1 [00:01<00:00,  1.37s/it]
LLMPageCorrectionProcessor running: 2it [00:02,  1.23s/it]                       

转换完成，输出文本前 400 字：


{0}------------------------------------------------

# 附录二 宋代建筑术语解释

## 一、检字

二十五画 攥

| 笔画   | 汉字                                        |
|------|-------------------------------------------|
| 二画   | 丁七八人九                                     |
| 三画   | 三土下大万上口山门小飞叉马子                            |
| 四画   | 切云天井木瓦五牙厅止日内仓分手牛乌勾丹月计方火斗心双水               |
| 五画   | 打平正布石龙出由四外卯令生白瓜汉立永对                   


## 保存pdf解析结果到本地

In [23]:
import os
from datetime import datetime

output_dir = r"knowledgeBase\pdfParsed_output"
os.makedirs(output_dir, exist_ok=True)

# ===== 2. 获取输入文件名（不含后缀）=====
input_filename = os.path.splitext(os.path.basename(input_pdf_path))[0]

# ===== 3. 生成时间戳，避免覆盖 =====
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ===== 4. 保存 Markdown 结果 =====
text_output_path = os.path.join(
    output_dir,
    f"{input_filename}_result_{timestamp}.md"
)

with open(text_output_path, "w", encoding="utf-8") as f:
    f.write(text)

# ===== 5. 保存图片（如果有）=====
image_output_dir = os.path.join(
    output_dir,
    f"{input_filename}_images_{timestamp}"
)

saved_images = []

if images:
    os.makedirs(image_output_dir, exist_ok=True)
    for idx, (image_name, image_data) in enumerate(images.items()):
        image_path = os.path.join(image_output_dir, image_name)

        # image_data 可能是 PIL Image 或 bytes
        if hasattr(image_data, "save"):  # PIL Image
            image_data.save(image_path)
        else:  # bytes
            with open(image_path, "wb") as img_f:
                img_f.write(image_data)

        saved_images.append(image_path)

# ===== 6. Notebook 输出说明 =====
print("✅ PDF 解析结果已保存")
print(f"📄 文本/markdown 文件: {text_output_path}")

if saved_images:
    print(f"图片目录: {image_output_dir}")
    print(f"图片数量: {len(saved_images)}")
else:
    print("本次解析未产生可导出的图片")


✅ PDF 解析结果已保存
📄 文本/markdown 文件: knowledgeBase\pdfParsed_output\《营造法式》解读 第2版术语库_test_result_20260126_193649.md
本次解析未产生可导出的图片


# mock chunks test totall pipline

In [ ]:
import json
from langchain_core.documents import Document

chunks_file_path = r"knowledgeBase\pdfParse\cleaned_data\rare_hanzi_integrated.jsonl"

langch_docs = []
with open(chunks_file_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        langch_docs.append(
            Document(
                page_content = obj["chunk_content"],
                metadata = obj["metadata"]
            )
        )


In [4]:
import pprint

print((len(langch_docs)))
pprint.pprint(langch_docs[11].page_content)
print(pprint.pformat(langch_docs[11].metadata))

print(langch_docs[11].metadata.get("chunk_id"))

501
'以上为《营造法式》(以下或简称《法式》)定性,说它是一种估算工料的定额,只是历史地、客观地阐明问题,丝毫没有贬低它的意思。在一个以儒家思想为正统的、文人士大夫不屑涉足工匠技艺的社会里,能出现这样一部反映当时建筑工程实况的巨著,是难能可贵的。加之李诫自身的条件——丰富的工程实践经验和勤奋努力的品格,使《营造法式》的编修达到了很高的水平,不愧为中国古代最完善的建筑专著和不朽的杰作,也是后世打开宋代建筑科学和艺术之门的一把钥匙。'
{'annotation': [],
 'book_info': '《营造法式》解读(2017年3月修订版) 潘谷西 何建中著',
 'chunk_id': 'yingzao_fashi_chap1_012',
 'chunk_size': 213,
 'closest_title': '（二）李诫《营造法式》的编写体例',
 'has_annotation': False,
 'has_image': False,
 'images': [],
 'toc_path': ['第一章  总论', '一、《营造法式》的性质与特点', '（二）李诫《营造法式》的编写体例']}
yingzao_fashi_chap1_012


In [ ]:
from langchain_community.embeddings import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model = "nomic-embed-text:latest",
    base_url = "http://localhost:11434"
)
print("Ollama embedding model loaded.")

# 从langch_docs提取文本内容进行嵌入计算
texts = [doc.page_content for doc in langch_docs]
print((len(texts)))
embeddings = embedding_model.embed_documents(texts)
# 将嵌入与对应的chunk_id和metadata关联起来
embedded_chunks = []
for doc, vector in zip(langch_docs, embeddings):
    embedded_chunks.append({
        "chunk_id": doc.metadata.get("chunk_id"),
        "embedding": vector,
        "metadata": doc.metadata
    })


In [ ]:
import os
import pprint
from openai import OpenAI
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain_community.embeddings import OpenAIEmbeddings
import numpy as np
from tqdm.notebook import tqdm  # Jupyter中更好的进度条
import time

# %% [markdown]
# ## 2. 配置API信息

# %%
# 设置API密钥（确保已经在环境变量中设置了DASHSCOPE_API_KEY）
API_KEY = os.getenv("DASHSCOPE_API_KEY")
if not API_KEY:
    raise ValueError("请设置环境变量 DASHSCOPE_API_KEY")

BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "text-embedding-v4"

print(f"使用模型: {MODEL_NAME}")
print(f"API Base URL: {BASE_URL}")

# %% [markdown]
# ## 3. 创建自定义的阿里云Embedding类

# %%
class AliyunEmbeddings:
    """
    阿里云文本向量模型的封装类，与LangChain兼容
    """
    def __init__(self, api_key, base_url, model_name):
        self.client = OpenAI(
            api_key=api_key,
            base_url=base_url
        )
        self.model_name = model_name
    
    def embed_documents(self, texts):
        """
        批量嵌入多个文档
        """
        try:
            completion = self.client.embeddings.create(
                model=self.model_name,
                input=texts
            )
            # 按原始顺序返回嵌入向量
            embeddings = [item.embedding for item in completion.data]
            return embeddings
        except Exception as e:
            print(f"批量嵌入失败: {e}")
            # 如果批量失败，回退到逐个处理
            return [self.embed_query(text) for text in texts]
    
    def embed_query(self, text):
        """
        嵌入单个查询文本
        """
        try:
            completion = self.client.embeddings.create(
                model=self.model_name,
                input=text
            )
            return completion.data[0].embedding
        except Exception as e:
            print(f"单个文本嵌入失败: {e}")
            # 返回零向量作为失败情况的占位符
            return [0.0] * 1536  # text-embedding-v4的维度

# %% [markdown]
# ## 4. 初始化embedding模型

# %%
# 创建embedding模型实例
embeddings_model = AliyunEmbeddings(
    api_key=API_KEY,
    base_url=BASE_URL,
    model_name=MODEL_NAME
)

# 测试embedding功能
try:
    test_embedding = embeddings_model.embed_query("测试文本")
    print(f"✓ Embedding模型初始化成功，向量维度: {len(test_embedding)}")
except Exception as e:
    print(f"✗ Embedding模型初始化失败: {e}")
    raise

# %% [markdown]
# ## 5. 准备文档数据

# %%
# 假设langch_docs已经在环境中存在
# 如果不在环境中，需要从其他地方加载
try:
    print(f"该库包含documents数量：{len(langch_docs)}")
    
    # 显示示例文档
    if len(langch_docs) > 11:
        print("\n示例文档（索引11）：")
        print(f"Document - Page Content: {langch_docs[11].page_content[:100]}...")
        print(f"Metadata: {pprint.pformat(langch_docs[11].metadata)}")
    
except NameError:
    print("错误：langch_docs 未定义，请先加载数据")
    # 这里可以添加从文件加载数据的代码
    raise

# %% [markdown]
# ## 6. 将所有文档转换为LangChain Document格式

# %%
# 将原始文档转换为LangChain的Document格式
documents = []
failed_docs = []  # 记录处理失败的文档

for i, doc in enumerate(tqdm(langch_docs, desc="转换文档格式")):
    try:
        # 确保page_content是字符串
        content = doc.page_content if hasattr(doc, 'page_content') else str(doc)
        metadata = doc.metadata if hasattr(doc, 'metadata') else {}
        
        # 添加索引信息到metadata
        metadata['index'] = i
        metadata['unicode'] = metadata.get('UNICODE', '')
        
        # 创建Document对象
        langchain_doc = Document(
            page_content=content,
            metadata=metadata
        )
        documents.append(langchain_doc)
    except Exception as e:
        failed_docs.append((i, str(e)))
        print(f"✗ 文档 {i} 转换失败: {e}")

print(f"\n文档转换完成: 成功 {len(documents)} 个, 失败 {len(failed_docs)} 个")
if failed_docs:
    print("失败的文档索引:")
    for idx, error in failed_docs[:5]:  # 只显示前5个
        print(f"  - 索引 {idx}: {error}")

# %% [markdown]
# ## 7. 批量处理向量化并存储到FAISS

# %%
# 设置FAISS存储路径
FAISS_INDEX_PATH = "./faiss_index_aliyun"

# 创建批量处理函数
def create_faiss_index_with_progress(documents, embeddings_model, batch_size=10):
    """
    分批处理文档并创建FAISS索引，显示进度
    """
    total_docs = len(documents)
    all_embeddings = []
    failed_embeddings = []
    
    print(f"\n开始处理 {total_docs} 个文档的向量化...")
    print(f"批量大小: {batch_size}")
    
    # 分批处理
    for i in range(0, total_docs, batch_size):
        batch = documents[i:i+batch_size]
        batch_texts = [doc.page_content for doc in batch]
        
        print(f"\n处理批次 {i//batch_size + 1}/{(total_docs-1)//batch_size + 1}")
        
        # 显示当前批次的文本预览
        for j, text in enumerate(batch_texts):
            preview = text[:50] + "..." if len(text) > 50 else text
            print(f"  文档 {i+j}: {preview}")
        
        # 获取向量
        try:
            batch_embeddings = embeddings_model.embed_documents(batch_texts)
            
            # 验证向量维度
            for emb in batch_embeddings:
                if len(emb) == 1536:  # text-embedding-v4的标准维度
                    all_embeddings.append(emb)
                else:
                    failed_embeddings.append((i + len(all_embeddings), "向量维度不正确"))
                    all_embeddings.append([0.0] * 1536)  # 添加零向量作为占位
                    
            print(f"  ✓ 批次处理成功，获得 {len(batch_embeddings)} 个向量")
            
        except Exception as e:
            print(f"  ✗ 批次处理失败: {e}")
            # 为这个批次的所有文档记录失败
            for idx in range(len(batch)):
                failed_embeddings.append((i + idx, str(e)))
                all_embeddings.append([0.0] * 1536)  # 添加零向量作为占位
        
        # 进度显示
        progress = (i + len(batch)) / total_docs * 100
        print(f"  总体进度: {progress:.1f}% ({i + len(batch)}/{total_docs})")
        
        # 避免API限流
        time.sleep(0.5)
    
    print(f"\n向量化完成: 成功 {len(all_embeddings) - len(failed_embeddings)} 个, 失败 {len(failed_embeddings)} 个")
    
    if failed_embeddings:
        print("\n失败的向量化记录:")
        for idx, error in failed_embeddings[:10]:  # 只显示前10个
            print(f"  - 文档 {idx}: {error}")
    
    return all_embeddings, failed_embeddings

# 执行向量化
embeddings_list, failed_list = create_faiss_index_with_progress(
    documents, 
    embeddings_model, 
    batch_size=10
)

# %% [markdown]
# ## 8. 创建并保存FAISS索引

# %%
print("\n开始创建FAISS索引...")

try:
    # 方法1：使用文本和向量直接创建
    # 由于FAISS.from_texts需要embedding函数，我们创建自定义的embedding类
    class SimpleEmbedding:
        def __init__(self, embeddings_list, documents):
            self.embeddings_list = embeddings_list
            self.documents = documents
            self.index_map = {doc.page_content: i for i, doc in enumerate(documents)}
        
        def embed_documents(self, texts):
            # 返回对应的向量
            return [self.embeddings_list[self.index_map[text]] for text in texts]
        
        def embed_query(self, text):
            # 查询时实时计算
            return embeddings_model.embed_query(text)
    
    # 创建简单的embedding包装器
    simple_embedding = SimpleEmbedding(embeddings_list, documents)
    
    # 创建FAISS索引
    vectorstore = FAISS.from_documents(
        documents=documents,
        embedding=simple_embedding
    )
    
    # 保存到本地
    vectorstore.save_local(FAISS_INDEX_PATH)
    print(f"✓ FAISS索引已保存到: {FAISS_INDEX_PATH}")
    
    # 显示索引信息
    print(f"索引中的文档数量: {vectorstore.index.ntotal}")
    
except Exception as e:
    print(f"✗ 创建FAISS索引失败: {e}")
    
    # 备用方法：手动创建索引
    print("\n尝试备用方法...")
    try:
        # 创建文档ID到向量的映射
        text_to_embedding = {}
        for i, (doc, emb) in enumerate(zip(documents, embeddings_list)):
            text_to_embedding[doc.page_content] = emb
        
        # 使用FAISS从零开始
        import faiss
        import pickle
        
        # 获取第一个向量的维度
        dimension = len(embeddings_list[0])
        
        # 创建索引
        index = faiss.IndexFlatL2(dimension)
        
        # 添加所有向量
        embeddings_array = np.array(embeddings_list).astype('float32')
        index.add(embeddings_array)
        
        # 保存索引和文档
        faiss.write_index(index, f"{FAISS_INDEX_PATH}/index.faiss")
        
        # 保存文档和元数据
        with open(f"{FAISS_INDEX_PATH}/index.pkl", 'wb') as f:
            pickle.dump((documents, text_to_embedding), f)
        
        print(f"✓ 使用备用方法保存索引成功")
        
    except Exception as e2:
        print(f"✗ 备用方法也失败: {e2}")

# %% [markdown]
# ## 9. 验证保存的索引

# %%
# 验证是否可以加载并使用索引
print("\n验证FAISS索引...")

try:
    # 尝试加载索引
    if os.path.exists(FAISS_INDEX_PATH):
        # 对于使用FAISS.from_documents保存的索引
        try:
            vectorstore = FAISS.load_local(
                FAISS_INDEX_PATH, 
                embeddings_model,
                allow_dangerous_deserialization=True
            )
            print("✓ 成功加载FAISS索引")
            
            # 测试相似性搜索
            test_query = langch_docs[11].page_content[:50]  # 使用第一个文档的前50个字符作为查询
            print(f"\n测试查询: {test_query}...")
            
            results = vectorstore.similarity_search(test_query, k=3)
            print("\n相似性搜索结果:")
            for i, doc in enumerate(results):
                print(f"\n结果 {i+1}:")
                print(f"  内容: {doc.page_content[:100]}...")
                print(f"  元数据: {doc.metadata}")
        except:
            print("无法使用标准方式加载，可能需要检查文件格式")
    else:
        print(f"索引目录不存在: {FAISS_INDEX_PATH}")
        
except Exception as e:
    print(f"验证过程中出现错误: {e}")

# %% [markdown]
# ## 10. 总结报告

# %%
print("\n" + "="*50)
print("处理总结报告")
print("="*50)

print(f"\n总文档数: {len(langch_docs)}")
print(f"成功转换文档: {len(documents)}")
print(f"文档转换失败: {len(failed_docs)}")
print(f"成功向量化: {len(embeddings_list) - len(failed_list)}")
print(f"向量化失败: {len(failed_list)}")
print(f"FAISS索引路径: {FAISS_INDEX_PATH}")

if os.path.exists(FAISS_INDEX_PATH):
    # 检查目录内容
    files = os.listdir(FAISS_INDEX_PATH)
    print(f"\n索引目录内容:")
    for file in files:
        size = os.path.getsize(os.path.join(FAISS_INDEX_PATH, file))
        print(f"  - {file} ({size/1024:.2f} KB)")

print("\n处理完成！")

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents = langch_docs,
    embedding = embedding_model
)
print("FAISS vectorstore created.")
print(f"Total vectors stored: {vectorstore.index.ntotal}")



FAISS vectorstore created.
Total vectors stored: 21


In [24]:
# 创建一个检索器，设置返回最相似的 chunk 数量
complex_filter = {
    "$and": [
        {"chapter_index": {"$eq": 1}},
        {"has_image": {"$eq": True}}
    ]
}

In [26]:
# ===== 9. 测试查询 =====
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3,
        "filter": complex_filter
    }
)
query = "《营造法式》与江南建筑的关系"
results = retriever.invoke(query)

print(f"Query: {query}")
print(f"Returned chunks: {len(results)}")
# ===== 10. 打印返回的 chunk 内容与元数据 =====
for i, doc in enumerate(results, start=1):
    print("=" * 80)
    print(f"[Result {i}]")
    print("Chunk ID:", doc.metadata.get("chunk_id"))
    print("Chapter:", doc.metadata.get("chapter"))
    print("Section:", doc.metadata.get("section"))
    print("Image:", doc.metadata.get("has_image"))
    print("Subsection:", doc.metadata.get("subsection"))
    print("Pages:", doc.metadata.get("page_start"), "-", doc.metadata.get("page_end"))
    print("-" * 40)
    print(doc.page_content[:200])  # 只打印前 500 字，防止刷屏


Query: 《营造法式》与江南建筑的关系
Returned chunks: 3
[Result 1]
Chunk ID: yingzao_2017_ch1_sum_0002
Chapter: 第一章 总论
Section: None
Image: True
Subsection: None
Pages: 4 - 12
----------------------------------------
《营造法式》与江南建筑有着密切的关系，其记载的拼柱法、竹材应用、串构件、蒜瓣柱与梭柱、上昂等做法，在江南地区宋代建筑中屡见不鲜。这些做法不仅美观，而且具有良好的力学性能，反映了江南建筑对自然材料的巧妙利用和技术创新。同时，江南与汴京之间的技术交流也促进了《法式》的编纂和推广。
[Result 2]
Chunk ID: yingzao_2017_ch1_sec2_0004
Chapter: 第一章 总论
Section: 二、《营造法式》与江南建筑的关系
Image: True
Subsection: None
Pages: 7 - 8
----------------------------------------
《营造法式》中记载的蒜瓣柱与梭柱，在江南地区宋代建筑中屡见不鲜，如苏州虎丘云岩寺塔、罗汉院大殿等。这些柱式不仅美观，而且具有良好的力学性能。然而，在北方地区却较为罕见，反映了南北建筑风格的差异。此外，上昂构件在江南地区的应用也极为广泛，且早于《法式》的记载，显示了江南建筑在技术创新方面的领先地位。
[Result 3]
Chunk ID: yingzao_2017_ch1_sec2_0001
Chapter: 第一章 总论
Section: 二、《营造法式》与江南建筑的关系
Image: True
Subsection: None
Pages: 4 - 5
----------------------------------------
浙江宁波保国寺大殿的拼柱法，是北宋时期拼合柱的孤例，与《营造法式》中记载的拼柱法不谋而合，反映了当时解决大料困难的智慧。此外，江南地区宋代建筑中的竹材使用、串构件、蒜瓣柱与梭柱、上昂等做法，与《营造法式》中的记载相印证，显示了江南建筑对《法式》的深远影响。这些做法在北方较为少见，进一步证明了《法式》与江南建筑的紧密联系。


In [27]:
# ===== 12. 查看相似度得分（调 chunk / embedding 用） =====
docs_with_scores = vectorstore.similarity_search_with_score(query, k=5)

for doc, score in docs_with_scores:
    print("=" * 60)
    print("Score:", score)
    print("Chunk ID:", doc.metadata.get("chunk_id"))
    print(doc.page_content[:300])


Score: 234.37265
Chunk ID: yingzao_2017_ch1_sum_0002
《营造法式》与江南建筑有着密切的关系，其记载的拼柱法、竹材应用、串构件、蒜瓣柱与梭柱、上昂等做法，在江南地区宋代建筑中屡见不鲜。这些做法不仅美观，而且具有良好的力学性能，反映了江南建筑对自然材料的巧妙利用和技术创新。同时，江南与汴京之间的技术交流也促进了《法式》的编纂和推广。
Score: 283.88727
Chunk ID: yingzao_2017_ch1_sec2_0004
《营造法式》中记载的蒜瓣柱与梭柱，在江南地区宋代建筑中屡见不鲜，如苏州虎丘云岩寺塔、罗汉院大殿等。这些柱式不仅美观，而且具有良好的力学性能。然而，在北方地区却较为罕见，反映了南北建筑风格的差异。此外，上昂构件在江南地区的应用也极为广泛，且早于《法式》的记载，显示了江南建筑在技术创新方面的领先地位。
Score: 286.09085
Chunk ID: yingzao_2017_ch1_sec2_0001
浙江宁波保国寺大殿的拼柱法，是北宋时期拼合柱的孤例，与《营造法式》中记载的拼柱法不谋而合，反映了当时解决大料困难的智慧。此外，江南地区宋代建筑中的竹材使用、串构件、蒜瓣柱与梭柱、上昂等做法，与《营造法式》中的记载相印证，显示了江南建筑对《法式》的深远影响。这些做法在北方较为少见，进一步证明了《法式》与江南建筑的紧密联系。
Score: 294.7055
Chunk ID: yingzao_2017_ch1_sec2_0005
《营造法式》中的“七朱八白”装饰，在江南五代至北宋间的建筑物上广泛应用，如杭州灵隐寺石塔、苏州虎丘塔等。这种装饰不仅美观，而且具有一定的文化意义。然而，在北方地区，虽然也有类似图案的出现，但在辽、金建筑上却未见此式，反映了南北建筑装饰风格的差异。此外，《法式》中的令栱长度规定也与江南建筑实践相符，进一步证明了《法式》与江南建筑的紧密联系。
Score: 358.60773
Chunk ID: yingzao_2017_ch1_sec2_0002
《营造法式》中竹材的广泛应用，不仅体现在殿阁、厅堂等高级建筑中，也深入到宫廷生活的各个方面。竹笆、竹篾编成的各种构件，如心柱编竹造、隔截编道、篾辫索等，展现了竹材在建筑中的多功能性。此外，竹材还用于壁画加固、临时凉棚等，

## 链接本地LLM，构造回答

In [28]:
from langchain_community.llms import Ollama

llm = Ollama(
    model="gemma3:latest", 
    base_url="http://localhost:11434",
    temperature=0.2
)
print("LLM loaded.")

e:\Miniconda\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM loaded.


C:\Users\孙俊强\AppData\Local\Temp\ipykernel_23004\3231409661.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
你是一名严谨的中国传统木建筑学术助理，请仅根据【参考资料】回答问题。

要求：
1. 回答应准确、学术化，使用专业术语
2. 不要编造资料中不存在的内容，如果资料中没有提及，则说明“不在参考资料中”
3. 回答末尾以【引用】形式列出使用到的 chunk_id toc_path和页码

【参考资料】
{context}

【问题】
{question}

【回答】
"""
)

In [46]:
def format_docs_with_citation(docs):
    formatted = []

    for doc in docs:
        metadata = doc.metadata or {}

        chunk_id = metadata.get("chunk_id", "unknown")
        page_start = metadata.get("page_start", "?")
        page_end = metadata.get("page_end", "?")
        toc_path = metadata.get("toc_path")

        if isinstance(toc_path, (list, tuple)):
            toc_path_str = ",".join(toc_path)
        elif isinstance(toc_path, str):
            toc_path_str = toc_path
        else:
            toc_path_str = "unknown"

        formatted.append(
            f"[chunk_id: {chunk_id}, toc_path: {toc_path_str}, pages: {page_start}-{page_end}]\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted)


In [43]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever | format_docs_with_citation,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)


In [54]:
query = "江南建筑和营造法式的关系"

answer = rag_chain.invoke(query)

print(answer)


《营造法式》与江南建筑有着密切的关系，其记载的拼柱法、竹材应用、串构件、蒜瓣柱与梭柱、上昂等做法，在江南地区宋代建筑中屡见不鲜。同时，江南与汴京之间的技术交流促进了《法式》的编纂和推广。在具体建筑中，如宁波保国寺的拼柱法、苏州虎丘云岩寺塔和罗汉院大殿中的蒜瓣柱与梭柱，以及江南地区广泛应用的上昂构件，都与《营造法式》的记载相印证，反映了江南建筑对《法式》的深远影响。【引用：yingzao_2017_ch1_sum_0002, 4-12; yingzao_2017_ch1_sec2_0001, 4-5; yingzao_2017_ch1_sec2_0004, 7-8】



### 以下为显示流程

In [52]:
query = "《营造法式》和江南建筑的关系是什么？"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} chunks.\n")

for i, doc in enumerate(retrieved_docs, start=1):
    print("=" * 80)
    print(f"[Chunk {i}]")
    print("chunk_id:", doc.metadata.get("chunk_id"))
    print("chapter:", doc.metadata.get("chapter"))
    print("pages:", doc.metadata.get("page_start"), "-", doc.metadata.get("page_end"))
    print("-" * 40)
    print(doc.page_content[:500]) 

Retrieved 3 chunks.

[Chunk 1]
chunk_id: yingzao_2017_ch1_sum_0002
chapter: 第一章 总论
pages: 4 - 12
----------------------------------------
《营造法式》与江南建筑有着密切的关系，其记载的拼柱法、竹材应用、串构件、蒜瓣柱与梭柱、上昂等做法，在江南地区宋代建筑中屡见不鲜。这些做法不仅美观，而且具有良好的力学性能，反映了江南建筑对自然材料的巧妙利用和技术创新。同时，江南与汴京之间的技术交流也促进了《法式》的编纂和推广。
[Chunk 2]
chunk_id: yingzao_2017_ch1_sec2_0001
chapter: 第一章 总论
pages: 4 - 5
----------------------------------------
浙江宁波保国寺大殿的拼柱法，是北宋时期拼合柱的孤例，与《营造法式》中记载的拼柱法不谋而合，反映了当时解决大料困难的智慧。此外，江南地区宋代建筑中的竹材使用、串构件、蒜瓣柱与梭柱、上昂等做法，与《营造法式》中的记载相印证，显示了江南建筑对《法式》的深远影响。这些做法在北方较为少见，进一步证明了《法式》与江南建筑的紧密联系。
[Chunk 3]
chunk_id: yingzao_2017_ch1_sec2_0004
chapter: 第一章 总论
pages: 7 - 8
----------------------------------------
《营造法式》中记载的蒜瓣柱与梭柱，在江南地区宋代建筑中屡见不鲜，如苏州虎丘云岩寺塔、罗汉院大殿等。这些柱式不仅美观，而且具有良好的力学性能。然而，在北方地区却较为罕见，反映了南北建筑风格的差异。此外，上昂构件在江南地区的应用也极为广泛，且早于《法式》的记载，显示了江南建筑在技术创新方面的领先地位。


In [53]:
# ===== 2. 构造 context（与 RAG Chain 中完全一致）=====
context_text = format_docs_with_citation(retrieved_docs)
print(len(context_text))
print("=" * 80)
print("Context passed to LLM:\n")
print(context_text)  # 防止过长


693
Context passed to LLM:

[chunk_id: yingzao_2017_ch1_sum_0002, toc_path: 第一章 总论, pages: 4-12]
《营造法式》与江南建筑有着密切的关系，其记载的拼柱法、竹材应用、串构件、蒜瓣柱与梭柱、上昂等做法，在江南地区宋代建筑中屡见不鲜。这些做法不仅美观，而且具有良好的力学性能，反映了江南建筑对自然材料的巧妙利用和技术创新。同时，江南与汴京之间的技术交流也促进了《法式》的编纂和推广。

[chunk_id: yingzao_2017_ch1_sec2_0001, toc_path: 第一章 总论,二、《营造法式》与江南建筑的关系, pages: 4-5]
浙江宁波保国寺大殿的拼柱法，是北宋时期拼合柱的孤例，与《营造法式》中记载的拼柱法不谋而合，反映了当时解决大料困难的智慧。此外，江南地区宋代建筑中的竹材使用、串构件、蒜瓣柱与梭柱、上昂等做法，与《营造法式》中的记载相印证，显示了江南建筑对《法式》的深远影响。这些做法在北方较为少见，进一步证明了《法式》与江南建筑的紧密联系。

[chunk_id: yingzao_2017_ch1_sec2_0004, toc_path: 第一章 总论,二、《营造法式》与江南建筑的关系, pages: 7-8]
《营造法式》中记载的蒜瓣柱与梭柱，在江南地区宋代建筑中屡见不鲜，如苏州虎丘云岩寺塔、罗汉院大殿等。这些柱式不仅美观，而且具有良好的力学性能。然而，在北方地区却较为罕见，反映了南北建筑风格的差异。此外，上昂构件在江南地区的应用也极为广泛，且早于《法式》的记载，显示了江南建筑在技术创新方面的领先地位。


In [50]:
# ===== 3. 显式调用 LLM =====
final_prompt = prompt.invoke({
    "context": context_text,
    "question": query
})

print("=" * 80)
print("Prompt sent to LLM:\n")
print(final_prompt)


Prompt sent to LLM:

messages=[HumanMessage(content='\n你是一名严谨的中国传统木建筑学术助理，请仅根据【参考资料】回答问题。\n\n要求：\n1. 回答应准确、简明、学术化，使用专业术语\n2. 不要编造资料中不存在的内容，如果资料中没有提及，则说明“不在参考资料中”\n3. 回答末尾以【引用】形式列出使用到的 chunk_id toc_path和页码\n\n【参考资料】\n[chunk_id: yingzao_2017_ch1_sec3_0003, toc_path: 第一章 总论,三、《营造法式》的内容取舍, pages: 14-15]\n在石作雕镌制度方面，《营造法式》详细记载了四种雕法，但在实际建筑中，雕刻品类更为丰富。例如，“实雕”和“平钑”两种雕刻方式，在实物中屡见不鲜，却未被《法式》收录。这可能是因为《法式》作为预算定额，更注重用料的规范性和经济性，而非全面记录雕刻艺术的多样性。这些遗漏为后世研究宋代建筑雕刻艺术提供了新的课题。\n\n[chunk_id: yingzao_2017_ch1_sum_0002, toc_path: 第一章 总论, pages: 4-12]\n《营造法式》与江南建筑有着密切的关系，其记载的拼柱法、竹材应用、串构件、蒜瓣柱与梭柱、上昂等做法，在江南地区宋代建筑中屡见不鲜。这些做法不仅美观，而且具有良好的力学性能，反映了江南建筑对自然材料的巧妙利用和技术创新。同时，江南与汴京之间的技术交流也促进了《法式》的编纂和推广。\n\n[chunk_id: yingzao_2017_ch1_sec2_0005, toc_path: 第一章 总论,二、《营造法式》与江南建筑的关系, pages: 10-11]\n《营造法式》中的“七朱八白”装饰，在江南五代至北宋间的建筑物上广泛应用，如杭州灵隐寺石塔、苏州虎丘塔等。这种装饰不仅美观，而且具有一定的文化意义。然而，在北方地区，虽然也有类似图案的出现，但在辽、金建筑上却未见此式，反映了南北建筑装饰风格的差异。此外，《法式》中的令栱长度规定也与江南建筑实践相符，进一步证明了《法式》与江南建筑的紧密联系。\n\n【问题】\n《营造法式》为什么被认为是一种建筑工程预算定额？\n\n【回答】\n', additional_kwargs={}, re

In [51]:
# ===== 4. 生成最终回答 =====
answer = llm.invoke(final_prompt)

print("=" * 80)
print("Final Answer:\n")
print(answer)


Final Answer:

《营造法式》被认为是一种建筑工程预算定额，是因为其更注重用料的规范性和经济性，而非全面记录雕刻艺术的多样性。 [chunk_id: yingzao_2017_ch1_sec3_0003, toc_path: 第一章 总论,三、《营造法式》的内容取舍, pages: 14-15]


# 对比rag和无rag回答

In [ ]:
def compare_rag_vs_no_rag(
    query: str,
    llm,
    retriever,
    prompt,
    format_docs_fn,
    top_k: int = 5
):
    """
    对同一个 query，对比：
    1. 无 RAG（纯 LLM）
    2. 有 RAG（Retriever + Context + LLM）

    参数说明：
    - llm: LangChain LLM 实例（与你 rag_chain 使用的同一个）
    - retriever: 向量检索器
    - prompt: ChatPromptTemplate（RAG 用）
    - format_docs_fn: 将 docs 转成 context 的函数
    """

    print("=" * 100)
    print("QUERY:")
    print(query)

    # ------------------------------------------------------------------
    # 1. 无 RAG（Baseline）
    # ------------------------------------------------------------------
    print("\n" + "=" * 100)
    print("BASELINE ANSWER (NO RAG):\n")

    baseline_answer = llm.invoke(query)
    print(baseline_answer)

    # ------------------------------------------------------------------
    # 2. 有 RAG：检索阶段
    # ------------------------------------------------------------------
    print("\n" + "=" * 100)
    print("RAG RETRIEVAL STAGE:\n")

    retrieved_docs = retriever.invoke(query)

    for i, doc in enumerate(retrieved_docs[:top_k], start=1):
        print("-" * 80)
        print(f"[Chunk {i}]")
        print("chunk_id:", doc.metadata.get("chunk_id"))
        print("chapter:", doc.metadata.get("chapter"))
        print("pages:", doc.metadata.get("page_start"), "-", doc.metadata.get("page_end"))
        print(doc.page_content[:400])

    # ------------------------------------------------------------------
    # 3. 构造 RAG Context
    # ------------------------------------------------------------------
    print("\n" + "=" * 100)
    print("RAG CONTEXT PASSED TO LLM:\n")

    context_text = format_docs_fn(retrieved_docs[:top_k])
    print(context_text[:2000])

    # ------------------------------------------------------------------
    # 4. 有 RAG：生成阶段
    # ------------------------------------------------------------------
    print("\n" + "=" * 100)
    print("RAG ANSWER:\n")

    rag_prompt = prompt.invoke({
        "context": context_text,
        "question": query
    })

    rag_answer = llm.invoke(rag_prompt)
    print(rag_answer)

    # ------------------------------------------------------------------
    # 5. 返回结构化结果（方便后续评估 / 写论文）
    # ------------------------------------------------------------------
    return {
        "query": query,
        "baseline_answer": baseline_answer,
        "rag_answer": rag_answer,
        "retrieved_chunks": retrieved_docs[:top_k]
    }


In [56]:
query = "江南建筑和营造法式的关系"

result = compare_rag_vs_no_rag(
    query=query,
    llm=llm,
    retriever=retriever,
    prompt=prompt,
    format_docs_fn=format_docs_with_citation,
    top_k=5
)


QUERY:
江南建筑和营造法式的关系

BASELINE ANSWER (NO RAG):

江南建筑和营造法式之间存在着复杂而密切的关系，它们在历史上相互影响、借鉴和发展，共同塑造了中国古代建筑的辉煌成就。要理解它们的关系，需要从以下几个方面进行分析：

**1. 营造法式：源头与基础**

* **起源：** 营造法式是唐朝时期由隋朝开始，在唐朝时期发展成熟的建筑理论体系和规范。它主要由《营造典范》和《营造实用典则》两部典籍构成。
* **核心思想：** 营造法式强调“以形为基，以理为本”，注重建筑的结构、材料、尺寸、比例、装饰等方面，力求达到和谐、稳定、美观的效果。它提出了“三尺为基”、“四尺为梁”、“五尺为柱”等基本尺寸标准，并对建筑的结构、材料、施工工艺等方面进行了详细的规定。
* **影响：** 营造法式对中国古代建筑产生了深远的影响，成为中国古代建筑理论和实践的基础。

**2. 江南建筑：继承与发展**

* **地理环境：** 江南地区地势平坦，水网密布，气候温和，非常适合建筑发展。同时，江南地区经济发达，工匠技术精湛，为建筑的发展提供了良好的条件。
* **继承营造法式：** 江南建筑在继承营造法式的基础上，进行了大量的创新和发展。它吸收了营造法式中的许多理论和技术，并根据江南地区的实际情况进行了调整和完善。
* **特点：** 江南建筑形成了独特的风格，主要特点包括：
    * **园林建筑：** 江南建筑最显著的特点就是园林建筑，如拙政园、留园、香山园等，这些园林以其精巧的设计、优美的景色、丰富的文化内涵而闻名于世。
    * **木结构建筑：** 江南建筑以木结构为主，注重木材的运用和加工，形成了独特的木结构建筑体系。
    * **装饰艺术：** 江南建筑的装饰艺术非常丰富，包括彩绘、雕刻、砖雕、石雕等，这些装饰艺术不仅美化了建筑，也反映了江南地区的文化和审美情趣。
    * **注重细节：** 江南建筑对细节的把握非常精细，如屋顶的坡度、柱子的尺寸、窗户的形状等，都经过精心设计和制作，体现了江南工匠的智慧和技艺。

**3. 相互影响与借鉴**

* **营造法式对江南建筑的影响：** 营造法式为江南建筑提供了理论基础和技术指导，江南建筑在继承营造法式的基础上，进行了大量的创新和发展。
* **江南建筑对营造法式的补充与完善：